In [4]:
project_root

'/hpc/group/coganlab/jz421/GlobalLocal/src'

In [8]:
# === Cell 1: imports & path ===
import os
import sys
try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from power_traces_anova_f_traces_vis import (
    plot_anova_F_traces_for_roi,
    plot_per_electrode_F_traces,
    plot_per_electrode_power_traces,
)
from src.analysis.power.power_traces import load_significant_electrodes

# Adjust to your data root
DERIV = os.path.join(project_root, 'dcc_scripts', 'power', 'figs')
EPOCHS_ROOT = 'Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit'
CONDITION_LABEL = 'stimulus_experiment_conditions'
RUN_DIR = os.path.join(DERIV, EPOCHS_ROOT, 'roi', CONDITION_LABEL)
ANOVA_DIR = os.path.join(RUN_DIR , 'anova_F_traces')
print('ANOVA files:', list(Path(ANOVA_DIR).glob('*.npz'))[:5])


ANOVA files: []


In [6]:
ANOVA_DIR

'/hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/power/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/roi/stimulus_lwpc_conditions/anova_F_traces'

In [ ]:

# === Cell 2: load metadata & one F-trace ===
meta = json.load(open(RUN_DIR / 'something_metadata.json'))   # replace with actual filename
print('factors:', meta['anova_factors'])
print('rois:', meta['rois'])

# === Cell 3: plot all F-traces for one ROI ===
fig = plot_anova_F_traces_for_roi(
    save_dir=RUN_DIR,
    roi='dlPFC',
    conditions_save_name='<conditions_save_name>',
)

# === Cell 4: within-electrode results & grid plot ===
within_run = RUN_DIR / 'within_elec_anova' / '<run_label>'
pages = plot_per_electrode_F_traces(
    within_run_dir=within_run,
    roi='dlPFC',
    effect='C(congruency)',
    channels_per_page=20,
)

# === Cell 5: filter sig electrodes & rerun a smaller pass ===
sig_elecs = load_significant_electrodes(within_run, roi='dlPFC', effect='C(congruency)')
print(f'{len(sig_elecs)} sig electrodes in dlPFC for C(congruency)')

# === Cell 6: latency comparison congruency vs switch ===
# Build (n_elec, n_times) F-trace arrays by stacking per-electrode npz files
def stack_f_traces(within_run, roi, effect):
    rows, names = [], []
    for f in sorted((Path(within_run) / roi).glob('*/*.npz')):
        d = np.load(f)
        ei = list(d['effect_names']).index(effect)
        rows.append(d['observed_F'][ei])
        names.append(f.stem)
    return np.stack(rows, axis=0), names, d['window_centers']

trc_cong,  names_c, times = stack_f_traces(within_run, 'dlPFC', 'C(congruency)')
trc_switch, _,       _    = stack_f_traces(within_run, 'dlPFC', 'C(switchType)')
result = compare_two_effects_latency_jackknife(trc_cong, trc_switch, times)
print('congruency latency:', result['lat_a'])
print('switchType latency:', result['lat_b'])
print('Miller t =', result['t'], 'p =', result['p'])